<a href="https://colab.research.google.com/github/Saberartoria77/equity-analytics-platform/blob/main/notebooks/rsi_significance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## RSI Mean-Reversion vs Buy-and-Hold: Significance Test
Tested on 100 S&P 500 stocks, 2021–2026. Position set on *prior-day* RSI(14)
(enter <30, exit >70) to avoid look-ahead. Metric: daily return difference
(strategy − market).

| Method | n | mean daily diff | p-value |
|---|---|---|---|
| Pooled stock-days (naive) | 115,743 | -0.000330 | 2e-09 |
| Daily cross-sectional mean (correct) | 1,166 | -0.000294 | 0.19 |

**Why they disagree:** pooling every stock-day treats ~100 stocks that move
together each day as independent (pseudo-replication), understating variance and
producing a spuriously tiny p. Averaging the diff to one value per day removes
the cross-sectional correlation; the honest test shows **no significant
difference** (p = 0.19).

**Conclusion:** RSI mean-reversion does not significantly beat or trail
buy-and-hold in daily returns. Cumulative gaps reflect reduced market exposure,
not a statistically reliable edge.

In [1]:
!pip install sqlalchemy psycopg2-binary pandas scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 26.8 MB/s eta 0:00:00


In [3]:
from getpass import getpass
from sqlalchemy import create_engine
import pandas as pd

DB_URL = getpass("Paste Railway DB_URL here: ")
engine = create_engine(DB_URL)

pd.read_sql("SELECT 1 AS ok", engine)

Paste Railway DB_URL here: ··········


,ok
0,1


In [4]:
tables = pd.read_sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""", engine)
print(tables)

# 挑几个核心表数数
for t in ["daily_prices", "indicators"]:
    try:
        n = pd.read_sql(f"SELECT COUNT(*) AS rows FROM {t}", engine)["rows"][0]
        print(f"{t}: {n} rows")
    except Exception as e:
        print(f"{t}: ⚠️ {e}")

                table_name
0             daily_prices
1       daily_returns_view
2               indicators
3           ingestion_runs
4  rolling_volatility_view
5                   stocks
daily_prices: 126243 rows
indicators: 115843 rows


Starting phase A

Is my RSI strategy really better than simple buy-and-hold, or is it just luck?
I am turning this into a hypothesis test:


$H_0$ (null hypothesis): There is no difference between the daily returns of the strategy and the daily returns of buy-and-hold (the mean of the difference = 0).


$H_1$: There is a difference.Method: Construct two daily return series (strategy vs. buy-and-hold), then test whether the gap between them is a 'true signal' or just 'noise'."

In [5]:
for t in ["daily_prices", "indicators"]:
    cols = pd.read_sql(f"""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = '{t}'
        ORDER BY ordinal_position
    """, engine)
    print(f"\n=== {t} ===")
    print(cols.to_string(index=False))

print("\n--- daily_prices 样本 ---")
print(pd.read_sql("SELECT * FROM daily_prices LIMIT 3", engine))
print("\n--- indicators 样本 ---")
print(pd.read_sql("SELECT * FROM indicators LIMIT 3", engine))


=== daily_prices ===
column_name data_type
         id   integer
   stock_id   integer
       date      date
       open   numeric
       high   numeric
        low   numeric
      close   numeric
     volume    bigint

=== indicators ===
   column_name        data_type
            id           bigint
      stock_id           bigint
          date             date
        sma_20 double precision
        sma_50 double precision
       sma_200 double precision
        rsi_14 double precision
     macd_line double precision
   macd_signal double precision
macd_histogram double precision
      bb_upper double precision
     bb_middle double precision
      bb_lower double precision

--- daily_prices 样本 ---
   id  stock_id        date      open      high       low     close     volume
0   1         2  2021-04-26  131.3238  131.5478  130.0868  131.2167   66905100
1   2         2  2021-04-27  131.4991  131.8887  130.6225  130.8952   66015800
2   3         2  2021-04-28  130.8173  131.5088  1

In [9]:
import numpy as np

STOCK_ID = 2

q = f"""
    SELECT p.date, p.close, i.rsi_14
    FROM daily_prices p
    JOIN indicators i
      ON p.stock_id = i.stock_id      -- joining both stock_id and date
     AND p.date     = i.date
    WHERE p.stock_id = {STOCK_ID}
    ORDER BY p.date
"""
df = pd.read_sql(q, engine)
df["close"] = df["close"].astype(float)


df["mkt_ret"] = df["close"].pct_change()


rsi_prev = df["rsi_14"].shift(1)


raw = pd.Series(np.nan, index=df.index)
raw[rsi_prev < 30] = 1.0
raw[rsi_prev > 70] = 0.0
df["position"] = raw.ffill().fillna(0.0)


df["strat_ret"] = df["mkt_ret"] * df["position"]

#check
print(df[["mkt_ret", "strat_ret"]].mean())
print("buy&hold count:", (1 + df["mkt_ret"]).prod() - 1)
print("strat overview:    ", (1 + df["strat_ret"]).prod() - 1)

mkt_ret      0.000754
strat_ret    0.000368
dtype: float64
buy&hold count: 1.0110699839993376
strat overview:     0.4025702642798459


We decided to a do t and p test to test out if our strat is worse or its buried in noises.

In [11]:
from scipy import stats


df["diff"] = df["strat_ret"] - df["mkt_ret"]
d = df["diff"].dropna()


t_stat, p_val = stats.ttest_1samp(d, popmean=0)
print(f"average difference: {d.mean():.6f}")
print(f"t = {t_stat:.3f},  p = {p_val:.4f}")

average difference: -0.000386
t = -1.061,  p = 0.2888


"p > 0.05, so we fail to reject H₀: there is not enough evidence to say there is a difference in daily returns between the strategy and buy-and-hold. In other words—+101% vs +40% look like worlds apart, but statistically, they are not significant. That daily average difference of -0.000386 was drowned out by huge daily volatility (the t-stat is only -1.06, far from ±2). Five years of compounding turned a statistically insignificant daily difference into an exaggerated-looking difference in terminal value.

Using bootstrap

In [12]:
import numpy as np

rng = np.random.default_rng(42)
diff_vals = d.values            # daily (strategy - market) differences
n = len(diff_vals)

# Resample with replacement 10,000 times, take the mean each time
boot_means = np.array([
    rng.choice(diff_vals, size=n, replace=True).mean()
    for _ in range(10_000)
])

lo, hi = np.percentile(boot_means, [2.5, 97.5])
print(f"Mean daily difference: {diff_vals.mean():.6f}")
print(f"Bootstrap 95% CI:      [{lo:.6f}, {hi:.6f}]")
print("Significant?", "NO - CI includes 0" if lo < 0 < hi else "YES - CI excludes 0")

Mean daily difference: -0.000386
Bootstrap 95% CI:      [-0.001079, 0.000336]
Significant? NO - CI includes 0


In the next process we will be expanding our data for testing to 100 stocks

In [13]:
import numpy as np
from scipy import stats

# 1) Pull close + RSI for ALL stocks
q_all = """
    SELECT p.stock_id, p.date, p.close, i.rsi_14
    FROM daily_prices p
    JOIN indicators i ON p.stock_id = i.stock_id AND p.date = i.date
    ORDER BY p.stock_id, p.date
"""
alldf = pd.read_sql(q_all, engine)
alldf["close"] = alldf["close"].astype(float)

# 2) Compute per stock (groupby so pct_change/shift never bleed across tickers)
def make_position(rsi_prev):
    raw = pd.Series(np.nan, index=rsi_prev.index)
    raw[rsi_prev < 30] = 1.0
    raw[rsi_prev > 70] = 0.0
    return raw.ffill().fillna(0.0)

grp = alldf.groupby("stock_id")
alldf["mkt_ret"]  = grp["close"].pct_change()
alldf["rsi_prev"] = grp["rsi_14"].shift(1)
alldf["position"] = alldf.groupby("stock_id")["rsi_prev"].transform(make_position)
alldf["strat_ret"] = alldf["position"] * alldf["mkt_ret"]
alldf["diff"] = alldf["strat_ret"] - alldf["mkt_ret"]
alldf = alldf.dropna(subset=["diff"])

# 3a) NAIVE: every stock-day as an independent observation (WRONG)
naive = alldf["diff"].values
t_n, p_n = stats.ttest_1samp(naive, 0)
print(f"[NAIVE pooled] n={len(naive):>6}  mean={naive.mean():.6f}  t={t_n:.2f}  p={p_n:.2e}")

# 3b) CORRECT: average diff across stocks each day, then test the daily series
daily = alldf.groupby("date")["diff"].mean()
t_d, p_d = stats.ttest_1samp(daily.values, 0)
print(f"[Daily mean]   n={len(daily):>6}  mean={daily.mean():.6f}  t={t_d:.2f}  p={p_d:.4f}")

[NAIVE pooled] n=115743  mean=-0.000330  t=-5.99  p=2.07e-09
[Daily mean]   n=  1166  mean=-0.000294  t=-1.32  p=0.1874
